# Performance Risk Report

Generate full-cohort pass/fail risk predictions and highlight the students with the highest fail risk and their primary risk factors.


In [6]:
import sys
from pathlib import Path

import pandas as pd

base_dir = Path('../..').resolve()
sys.path.append(str(base_dir))

from src.predict import load_combined_performance_data, load_model, predict_students

processed_dir = base_dir / 'data' / 'processed' / 'performance'
model_path = base_dir / 'models' / 'performance' / 'pass_classifier_rf.joblib'


In [7]:
data = load_combined_performance_data(base_dir)
feature_df = data.drop(columns=['G3', 'pass'])
model = load_model(model_path)
report_df = predict_students(feature_df, model=model)
report_df.insert(0, 'student_id', range(1, len(report_df) + 1))
report_df['actual_pass'] = data['pass'].reset_index(drop=True)
report_df['G3'] = data['G3'].reset_index(drop=True)
report_df['predicted_outcome'] = report_df['prediction'].map({0: 'Fail', 1: 'Pass'})
report_df = report_df.sort_values(by='risk_score', ascending=False).reset_index(drop=True)
report_df.head()


,student_id,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,...,G1,G2,dataset,prediction,risk_score,risk_level,Primary_Risk_Factors,actual_pass,G3,predicted_outcome
0,129,GP,M,18,R,GT3,T,2,2,services,...,7,4,math,0,0.955466,High,Low G2 Score | Low G1 Score | Prior Class Fail...,0,0,Fail
1,19,GP,M,17,U,GT3,T,3,2,services,...,6,5,math,0,0.949535,High,Low G2 Score | Low G1 Score | Prior Class Fail...,0,5,Fail
2,161,GP,M,17,R,LE3,T,2,1,at_home,...,7,6,math,0,0.948779,High,Low G2 Score | Low G1 Score | Prior Class Fail...,0,0,Fail
3,145,GP,M,17,U,GT3,T,2,1,other,...,5,0,math,0,0.946895,High,Low G2 Score | Low G1 Score | Prior Class Fail...,0,0,Fail
4,151,GP,M,18,U,LE3,T,1,1,other,...,6,5,math,0,0.945214,High,Low G2 Score | Low G1 Score | Prior Class Fail...,0,0,Fail


In [8]:
output_path = processed_dir / 'student_predictions.csv'
report_df.to_csv(output_path, index=False)
print(f'Saved full performance risk report to: {output_path}')


Saved full performance risk report to: /Users/admin/Documents/GitHub/Student Dropout Prediction and Alert System/data/processed/performance/student_predictions.csv


## Risk distribution summary

- Risk distribution in the saved report:
- Low risk: 753
- Medium risk: 66
- High risk: 225
- Common risk explanations include low previous grades, low study time, high social time, and high absences.


In [9]:
risk_distribution = (
    report_df['risk_level']
    .value_counts()
    .reindex(['Low', 'Medium', 'High'])
    .fillna(0)
    .astype(int)
    .rename_axis('risk_level')
    .reset_index(name='student_count')
)

common_risk_explanations = report_df['Primary_Risk_Factors'].value_counts().head(10)

print('Risk distribution in saved report')
display(risk_distribution)
print('Most common primary risk explanations')
display(common_risk_explanations)


Risk distribution in saved report


,risk_level,student_count
0,Low,753
1,Medium,66
2,High,225


Most common primary risk explanations


Primary_Risk_Factors
General Academic Risk Profile                           278
Low G2 Score | Low G1 Score | Prior Class Failures      106
High Social Time                                         91
Low Study Time                                           73
Low G2 Score | Low G1 Score | Low Study Time             41
High Social Time | High Alcohol Use                      35
Low Study Time | High Social Time | High Alcohol Use     31
Low G2 Score | Low G1 Score | High Social Time           30
Low G2 Score | Low G1 Score                              27
High Absences                                            23
Name: count, dtype: int64

In [10]:
risk_columns = [
    'student_id',
    'predicted_outcome',
    'risk_score',
    'risk_level',
    'Primary_Risk_Factors',
    'G1',
    'G2',
    'G3',
    'absences',
    'studytime',
]
report_df.loc[report_df['risk_level'].isin(['High', 'Medium']), risk_columns].head(25)


,student_id,predicted_outcome,risk_score,risk_level,Primary_Risk_Factors,G1,G2,G3,absences,studytime
0,129,Fail,0.955466,High,Low G2 Score | Low G1 Score | Prior Class Fail...,7,4,0,0,1
1,19,Fail,0.949535,High,Low G2 Score | Low G1 Score | Prior Class Fail...,6,5,5,16,1
2,161,Fail,0.948779,High,Low G2 Score | Low G1 Score | Prior Class Fail...,7,6,0,0,1
3,145,Fail,0.946895,High,Low G2 Score | Low G1 Score | Prior Class Fail...,5,0,0,0,1
4,151,Fail,0.945214,High,Low G2 Score | Low G1 Score | Prior Class Fail...,6,5,0,0,1
5,91,Fail,0.944070,High,Low G2 Score | Low G1 Score,7,7,8,0,3
6,214,Fail,0.942492,High,Low G2 Score | Low G1 Score | Prior Class Fail...,6,7,8,15,2
7,222,Fail,0.942302,High,Low G2 Score | Low G1 Score | Prior Class Fail...,6,5,0,0,3
8,174,Fail,0.939235,High,Low G2 Score | Low G1 Score | Prior Class Fail...,8,7,0,0,2
9,147,Fail,0.938415,High,Low G2 Score | Low G1 Score | Prior Class Fail...,6,7,0,0,2
